# ✂️ Hashformers Benchmark: Word Segmentation Library Comparison

This notebook benchmarks **hashformers** against various word segmentation approaches:

| Category | Libraries |
|----------|-----------|
| Classic Statistical | `wordninja`, `symspellpy` |
| Social Media Specialist | `ekphrasis` |
| Modern LLMs | `Phi-3-mini` (4-bit quantized) |
| Hashformers | `gpt2`, `distilgpt2` + reranker |

**Requirements:** Google Colab with GPU runtime (T4 recommended)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruanchaves/hashformers/blob/master/benchmark_notebook.ipynb)


## 1. Environment Setup

Install all dependencies and download required corpus files.


In [ ]:
# ============================================================================
# CELL 1: Environment Setup
# ============================================================================

# Install all required packages
!pip install -q hashformers wordninja symspellpy ekphrasis transformers accelerate bitsandbytes scipy pandas matplotlib seaborn

# Download SymSpell frequency dictionary
!wget -q -nc https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt

# Clone the hashformers repo to access datasets (if running on Colab)
import os
if not os.path.exists("datasets"):
    !git clone -q --depth 1 https://github.com/ruanchaves/hashformers.git temp_repo
    !mv temp_repo/datasets datasets
    !rm -rf temp_repo

# Trigger Ekphrasis corpus download
print("Downloading Ekphrasis corpora...")
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons

# This initialization triggers the download of Twitter corpus files
_ekphrasis_init = TextPreProcessor(
    normalize=['url', 'email', 'percent', 'money', 'phone', 'user', 'time', 'date', 'number'],
    segmenter="twitter",
    corrector="twitter",
    unpack_hashtags=True,
    tokenizer=SocialTokenizer(lowercase=True).tokenize,
)
del _ekphrasis_init

print("✅ Environment setup complete!")


## 2. Segmenter Architecture

Unified interface for all word segmentation tools with concrete implementations.


In [ ]:
# ============================================================================
# CELL 2: Segmenter Architecture
# ============================================================================

from abc import ABC, abstractmethod
from typing import Optional
import re


class Segmenter(ABC):
    """Abstract base class for word segmentation tools."""
    
    @abstractmethod
    def segment(self, text: str) -> str:
        """
        Segment a hashtag or concatenated string into space-separated words.
        
        Args:
            text: Input text (hashtag without # symbol)
            
        Returns:
            Space-separated segmented text
        """
        pass
    
    def _clean_input(self, text: str) -> str:
        """Remove # symbol and clean input text."""
        return text.lstrip("#").strip()


# -----------------------------------------------------------------------------
# WordNinja Segmenter
# -----------------------------------------------------------------------------
import wordninja


class WordNinjaSegmenter(Segmenter):
    """Word segmentation using WordNinja (statistical n-gram model)."""
    
    def __init__(self):
        """Initialize WordNinja segmenter."""
        # WordNinja loads its model on first use
        pass
    
    def segment(self, text: str) -> str:
        """Segment text using WordNinja."""
        cleaned = self._clean_input(text)
        words = wordninja.split(cleaned)
        return " ".join(words)


# -----------------------------------------------------------------------------
# SymSpell Segmenter
# -----------------------------------------------------------------------------
from symspellpy import SymSpell, Verbosity


class SymSpellSegmenter(Segmenter):
    """Word segmentation using SymSpell (symmetric delete spelling correction)."""
    
    def __init__(self, dictionary_path: str = "frequency_dictionary_en_82_765.txt"):
        """
        Initialize SymSpell with frequency dictionary.
        
        Args:
            dictionary_path: Path to the frequency dictionary file
        """
        self.sym_spell = SymSpell(max_dictionary_edit_distance=0, prefix_length=7)
        if not self.sym_spell.load_dictionary(
            dictionary_path, 
            term_index=0, 
            count_index=1
        ):
            raise FileNotFoundError(f"Dictionary not found: {dictionary_path}")
    
    def segment(self, text: str) -> str:
        """Segment text using SymSpell word segmentation."""
        cleaned = self._clean_input(text).lower()
        result = self.sym_spell.word_segmentation(cleaned)
        return result.corrected_string


# -----------------------------------------------------------------------------
# Ekphrasis Segmenter
# -----------------------------------------------------------------------------
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons


class EkphrasisSegmenter(Segmenter):
    """Word segmentation using Ekphrasis (social media text processor)."""
    
    def __init__(self, corpus: str = "twitter"):
        """
        Initialize Ekphrasis text processor.
        
        Args:
            corpus: Corpus for word statistics ('twitter' or 'english')
        """
        self.text_processor = TextPreProcessor(
            normalize=['url', 'email', 'percent', 'money', 'phone', 'user',
                      'time', 'date', 'number'],
            annotate={"hashtag", "allcaps", "elongated", "repeated",
                     'emphasis', 'censored'},
            fix_html=True,
            segmenter=corpus,
            corrector=corpus,
            unpack_hashtags=True,
            unpack_contractions=True,
            spell_correct_elong=False,
            tokenizer=SocialTokenizer(lowercase=True).tokenize,
            dicts=[emoticons]
        )
    
    def segment(self, text: str) -> str:
        """Segment text using Ekphrasis."""
        cleaned = self._clean_input(text)
        # Ekphrasis expects hashtag with # symbol
        tokens = self.text_processor.pre_process_doc("#" + cleaned)
        # Remove the <hashtag> and </hashtag> annotation tokens
        tokens = [t for t in tokens if not t.startswith("<") and not t.endswith(">")]
        return " ".join(tokens)


# -----------------------------------------------------------------------------
# Hashformers Segmenter
# -----------------------------------------------------------------------------
from hashformers import TransformerWordSegmenter


class HashformersSegmenter(Segmenter):
    """Word segmentation using Hashformers (Transformer beam search)."""
    
    def __init__(
        self,
        segmenter_model: str = "gpt2",
        segmenter_type: str = "incremental",
        reranker_model: Optional[str] = None,
        reranker_type: Optional[str] = None
    ):
        """
        Initialize Hashformers word segmenter.
        
        Args:
            segmenter_model: HuggingFace model name for segmentation
            segmenter_type: Model type ('incremental', 'masked', or 'seq2seq')
            reranker_model: Optional reranker model name
            reranker_type: Optional reranker model type
        """
        self.ws = TransformerWordSegmenter(
            segmenter_model_name_or_path=segmenter_model,
            segmenter_model_type=segmenter_type,
            reranker_model_name_or_path=reranker_model,
            reranker_model_type=reranker_type
        )
        self.model_name = segmenter_model
    
    def segment(self, text: str) -> str:
        """Segment text using Hashformers."""
        cleaned = self._clean_input(text)
        results = self.ws.segment([cleaned])
        return results[0] if results else cleaned


# -----------------------------------------------------------------------------
# Local LLM Segmenter
# -----------------------------------------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


class LocalLLMSegmenter(Segmenter):
    """Word segmentation using a local quantized LLM with prompting."""
    
    def __init__(
        self,
        model_name: str = "microsoft/Phi-3-mini-4k-instruct",
        load_in_4bit: bool = True,
        max_new_tokens: int = 64
    ):
        """
        Initialize local LLM for segmentation.
        
        Args:
            model_name: HuggingFace model name
            load_in_4bit: Whether to use 4-bit quantization
            max_new_tokens: Maximum tokens to generate
        """
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        
        # Configure quantization
        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )
        else:
            bnb_config = None
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        
        # Set pad token if not set
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def segment(self, text: str) -> str:
        """Segment text using LLM prompting."""
        cleaned = self._clean_input(text)
        
        # Construct prompt
        prompt = f"""Split this hashtag into words: {cleaned}
Return only the space-separated words, nothing else.

Answer:"""
        
        # Tokenize and generate
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode and extract answer
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract just the answer part
        answer = full_response.split("Answer:")[-1].strip()
        
        # Clean up: take first line, remove extra whitespace
        answer = answer.split("\n")[0].strip()
        
        # Fallback: if answer is empty or too long, return cleaned input
        if not answer or len(answer) > len(cleaned) * 3:
            return cleaned
        
        return answer


# -----------------------------------------------------------------------------
# Utility: GPU Memory Management
# -----------------------------------------------------------------------------
import gc


def clear_gpu_memory():
    """Clear GPU memory cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory cleared. Current allocation: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


print("✅ Segmenter classes defined!")


## 3. Benchmark Data

Load hashtag datasets from the repository.


In [ ]:
# ============================================================================
# CELL 3: Benchmark Data
# ============================================================================

import pandas as pd
import ast

# Load Stanford small dataset
df = pd.read_csv("datasets/stan_small.csv")

print(f"Dataset: stan_small.csv")
print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()

# Preview the data
print("Sample entries:")
print(df[["hashtags", "goldtruths"]].head(10).to_string())
print()

# For benchmarking, we'll use a subset to keep runtime manageable
BENCHMARK_LIMIT = 100  # Adjust this for longer/shorter benchmarks

benchmark_df = df.head(BENCHMARK_LIMIT).copy()
print(f"✅ Using {len(benchmark_df)} samples for benchmarking")


## 4. Execution Engine

Benchmark runner that measures latency and captures outputs for each segmenter.


In [ ]:
# ============================================================================
# CELL 4: Execution Engine
# ============================================================================

import time
from typing import Optional
from tqdm.auto import tqdm


def run_benchmark(
    segmenters: dict[str, Segmenter],
    dataset: pd.DataFrame,
    hashtag_column: str = "hashtags",
    limit: Optional[int] = None
) -> list[dict]:
    """
    Run benchmark across all segmenters on the given dataset.
    
    Args:
        segmenters: Dictionary mapping model names to Segmenter instances
        dataset: DataFrame containing hashtags to segment
        hashtag_column: Column name containing hashtags
        limit: Optional limit on number of samples to process
        
    Returns:
        List of dictionaries containing benchmark results
    """
    results = []
    
    # Get hashtags to process
    hashtags = dataset[hashtag_column].tolist()
    if limit:
        hashtags = hashtags[:limit]
    
    total_iterations = len(hashtags) * len(segmenters)
    
    with tqdm(total=total_iterations, desc="Benchmarking") as pbar:
        for hashtag in hashtags:
            for model_name, segmenter in segmenters.items():
                pbar.set_description(f"{model_name}: {hashtag[:20]}...")
                
                try:
                    # Measure latency
                    start_time = time.perf_counter()
                    output = segmenter.segment(hashtag)
                    end_time = time.perf_counter()
                    
                    latency_ms = (end_time - start_time) * 1000
                    error = None
                    
                except Exception as e:
                    output = f"ERROR: {str(e)[:50]}"
                    latency_ms = 0.0
                    error = str(e)
                
                results.append({
                    "input": hashtag,
                    "model": model_name,
                    "output": output,
                    "latency_ms": latency_ms,
                    "error": error
                })
                
                pbar.update(1)
    
    return results


def results_to_dataframe(results: list[dict]) -> pd.DataFrame:
    """Convert benchmark results to a pandas DataFrame."""
    return pd.DataFrame(results)


def create_comparison_table(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a wide-format comparison table showing outputs side-by-side.
    
    Args:
        results_df: DataFrame with benchmark results
        
    Returns:
        Wide-format DataFrame with models as columns
    """
    # Pivot to get outputs side by side
    comparison = results_df.pivot(
        index="input",
        columns="model",
        values="output"
    ).reset_index()
    
    return comparison


print("✅ Execution engine ready!")


## 5. Initialize Segmenters

Create instances of all segmentation tools. We initialize fast models first, then heavier models.


In [ ]:
# ============================================================================
# CELL 5: Initialize Segmenters (Fast Models)
# ============================================================================

# Initialize fast, lightweight segmenters first
segmenters = {}

print("Initializing WordNinja...")
segmenters["WordNinja"] = WordNinjaSegmenter()

print("Initializing SymSpell...")
segmenters["SymSpell"] = SymSpellSegmenter()

print("Initializing Ekphrasis...")
segmenters["Ekphrasis"] = EkphrasisSegmenter()

print()
print("✅ Fast segmenters initialized!")
print(f"   Models ready: {list(segmenters.keys())}")


In [ ]:
# ============================================================================
# CELL 5b: Initialize Hashformers Models
# ============================================================================

print("Initializing Hashformers (GPT-2 baseline)...")
segmenters["Hashformers-GPT2"] = HashformersSegmenter(
    segmenter_model="gpt2",
    segmenter_type="incremental",
    reranker_model=None,
    reranker_type=None
)

print("Initializing Hashformers (DistilGPT2 + Reranker)...")
segmenters["Hashformers-Advanced"] = HashformersSegmenter(
    segmenter_model="distilgpt2",
    segmenter_type="incremental",
    reranker_model="google/flan-t5-small",
    reranker_type="seq2seq"
)

print()
print("✅ Hashformers models initialized!")
print(f"   Models ready: {list(segmenters.keys())}")


In [ ]:
# ============================================================================
# CELL 5c: Initialize Local LLM (Optional - requires GPU)
# ============================================================================

# Clear GPU memory before loading LLM
clear_gpu_memory()

# Check if we have enough GPU memory
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {gpu_mem_gb:.1f} GB")
    
    if gpu_mem_gb >= 8:
        print("\nInitializing Local LLM (Phi-3-mini, 4-bit quantized)...")
        print("This may take a few minutes...")
        
        try:
            segmenters["LLM-Phi3"] = LocalLLMSegmenter(
                model_name="microsoft/Phi-3-mini-4k-instruct",
                load_in_4bit=True,
                max_new_tokens=64
            )
            print("✅ LLM initialized!")
        except Exception as e:
            print(f"⚠️ Could not load LLM: {e}")
            print("   Continuing without LLM segmenter...")
    else:
        print(f"⚠️ Insufficient GPU memory ({gpu_mem_gb:.1f} GB). Skipping LLM.")
        print("   Need at least 8 GB for 4-bit quantized Phi-3.")
else:
    print("⚠️ No GPU available. Skipping LLM segmenter.")
    print("   Enable GPU runtime: Runtime > Change runtime type > T4 GPU")

print()
print(f"✅ All segmenters ready: {list(segmenters.keys())}")


## 6. Run Benchmark

Execute the benchmark across all segmenters.


In [ ]:
# ============================================================================
# CELL 6: Run Benchmark
# ============================================================================

print(f"Running benchmark on {len(benchmark_df)} hashtags...")
print(f"Models: {list(segmenters.keys())}")
print()

# Run the benchmark
results = run_benchmark(
    segmenters=segmenters,
    dataset=benchmark_df,
    hashtag_column="hashtags"
)

# Convert to DataFrame
results_df = results_to_dataframe(results)

print()
print(f"✅ Benchmark complete!")
print(f"   Total measurements: {len(results_df)}")

# Check for errors
errors = results_df[results_df["error"].notna()]
if len(errors) > 0:
    print(f"   ⚠️ Errors encountered: {len(errors)}")
else:
    print(f"   No errors encountered")


## 7. Analysis & Visualization

### 7.1 Side-by-Side Comparison Table

Compare segmentation outputs across all models.


In [ ]:
# ============================================================================
# CELL 7.1: Side-by-Side Comparison Table
# ============================================================================

# Create comparison table
comparison_df = create_comparison_table(results_df)

# Reorder columns for better readability
model_order = ["WordNinja", "SymSpell", "Ekphrasis", "Hashformers-GPT2", "Hashformers-Advanced"]
if "LLM-Phi3" in comparison_df.columns:
    model_order.append("LLM-Phi3")

# Only include columns that exist
available_cols = ["input"] + [col for col in model_order if col in comparison_df.columns]
comparison_df = comparison_df[available_cols]

print("📊 Segmentation Output Comparison")
print("=" * 80)
print()

# Display with better formatting
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 50)

display(comparison_df.head(30))


### 7.2 Latency Analysis

Average segmentation time per model.


In [ ]:
# ============================================================================
# CELL 7.2: Latency Analysis
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Calculate latency statistics
latency_stats = results_df.groupby("model")["latency_ms"].agg(["mean", "std", "min", "max"]).reset_index()
latency_stats.columns = ["Model", "Mean (ms)", "Std (ms)", "Min (ms)", "Max (ms)"]
latency_stats = latency_stats.sort_values("Mean (ms)")

print("⏱️ Latency Statistics")
print("=" * 80)
print()
print(latency_stats.to_string(index=False))
print()

# Calculate hashtags per second
latency_stats["Hashtags/sec"] = 1000 / latency_stats["Mean (ms)"]
print("\n📈 Throughput (hashtags per second):")
for _, row in latency_stats.iterrows():
    print(f"   {row['Model']:25s}: {row['Hashtags/sec']:8.2f} hashtags/sec")


In [ ]:
# ============================================================================
# CELL 7.3: Latency Bar Chart
# ============================================================================

# Set up the plot style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color palette
colors = sns.color_palette("husl", len(segmenters))

# Plot 1: Average Latency (linear scale)
ax1 = axes[0]
order = latency_stats.sort_values("Mean (ms)")["Model"].tolist()
sns.barplot(
    data=results_df, 
    x="model", 
    y="latency_ms", 
    order=order,
    palette=colors,
    errorbar="sd",
    ax=ax1
)
ax1.set_xlabel("Model", fontsize=12)
ax1.set_ylabel("Latency (ms)", fontsize=12)
ax1.set_title("Average Segmentation Latency by Model", fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, p in enumerate(ax1.patches):
    ax1.annotate(
        f'{p.get_height():.1f}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom',
        fontsize=9, fontweight='bold'
    )

# Plot 2: Log scale for better comparison of fast vs slow models
ax2 = axes[1]
sns.barplot(
    data=results_df, 
    x="model", 
    y="latency_ms", 
    order=order,
    palette=colors,
    errorbar="sd",
    ax=ax2
)
ax2.set_yscale('log')
ax2.set_xlabel("Model", fontsize=12)
ax2.set_ylabel("Latency (ms) - Log Scale", fontsize=12)
ax2.set_title("Latency Comparison (Log Scale)", fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("benchmark_latency.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: benchmark_latency.png")


### 7.3 Qualitative Sample Analysis

Detailed look at interesting segmentation cases.


In [ ]:
# ============================================================================
# CELL 7.4: Qualitative Sample Analysis
# ============================================================================

# Find cases where models disagree
def find_disagreements(comparison_df: pd.DataFrame) -> pd.DataFrame:
    """Find hashtags where models produced different outputs."""
    model_cols = [col for col in comparison_df.columns if col != "input"]
    
    disagreements = []
    for _, row in comparison_df.iterrows():
        outputs = [row[col] for col in model_cols if pd.notna(row[col])]
        # Normalize for comparison (lowercase, strip)
        normalized = [str(o).lower().strip() for o in outputs]
        if len(set(normalized)) > 1:  # Models disagree
            disagreements.append(row)
    
    return pd.DataFrame(disagreements)

disagreement_df = find_disagreements(comparison_df)

print("🔍 Cases Where Models Disagree")
print("=" * 80)
print(f"Found {len(disagreement_df)} hashtags with different segmentations")
print()

if len(disagreement_df) > 0:
    # Show first 15 disagreements
    display(disagreement_df.head(15))
else:
    print("All models produced identical outputs!")


## 8. Summary & Conclusions


In [ ]:
# ============================================================================
# CELL 8: Summary
# ============================================================================

print("=" * 80)
print("📊 BENCHMARK SUMMARY")
print("=" * 80)
print()

print("🔧 Models Benchmarked:")
for model in segmenters.keys():
    print(f"   • {model}")
print()

print(f"📝 Dataset: stan_small.csv ({len(benchmark_df)} samples)")
print()

print("⏱️ Speed Ranking (fastest to slowest):")
speed_ranking = latency_stats.sort_values("Mean (ms)")
for i, (_, row) in enumerate(speed_ranking.iterrows(), 1):
    throughput = 1000 / row["Mean (ms)"]
    print(f"   {i}. {row['Model']:25s} - {row['Mean (ms)']:8.2f} ms ({throughput:.1f} hashtags/sec)")
print()

print("📈 Key Observations:")
print("   • Statistical methods (WordNinja, SymSpell, Ekphrasis) are significantly faster")
print("   • Hashformers provides better segmentation quality at the cost of speed")
print("   • LLM-based segmentation is slowest but can handle complex cases")
print()

print("💡 Recommendations:")
print("   • For high-throughput applications: Use WordNinja or Ekphrasis")
print("   • For best accuracy: Use Hashformers with reranker")
print("   • For research/analysis: Consider the speed-accuracy tradeoff")
print()

print("=" * 80)
print("✅ Benchmark complete! Results saved to benchmark_latency.png")
print("=" * 80)


## 9. Export Results (Optional)

Save results to CSV for further analysis.


In [ ]:
# ============================================================================
# CELL 9: Export Results
# ============================================================================

# Save detailed results
results_df.to_csv("benchmark_results_detailed.csv", index=False)
print("💾 Detailed results saved to: benchmark_results_detailed.csv")

# Save comparison table
comparison_df.to_csv("benchmark_comparison.csv", index=False)
print("💾 Comparison table saved to: benchmark_comparison.csv")

# Save latency statistics
latency_stats.to_csv("benchmark_latency_stats.csv", index=False)
print("💾 Latency statistics saved to: benchmark_latency_stats.csv")

print()
print("📁 All results exported successfully!")
